In [ ]:
import os
import gc
import warnings
import logging
import time
import math
import cv2
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from tqdm.auto import tqdm
import torchvision.transforms.v2 as transforms_v2

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

class CFG:
    test_soundscapes = '/kaggle/input/birdclef-2025/test_soundscapes'
    submission_csv = '/kaggle/input/birdclef-2025/sample_submission.csv'
    taxonomy_csv = '/kaggle/input/birdclef-2025/taxonomy.csv'
    model_path1 = '/kaggle/input/modelwithunlabeldata/efficientnet_b0_best (2).pth'
    model_path2 = '/kaggle/input/datacuatuantran/efficientnet_b0.pth'
    
    sample_rate = 32000
    WINDOW_SIZE = 5
    
    n_mels = 256
    fmin = 20
    fmax = 16000
    n_fft = 2048
    hop_length = 512
    power = 2.0
    norm = 'slaney'
    pad_mode = 'constant'
    top_db = 80
    target_shape = (256, 256)
    
    model_names = ['efficientnet_b0']  
    in_channels = 3  # RGB 3 kênh
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    use_tta = False
    tta_count = 3
    threshold = 0.7
    
    use_specific_folds = False
    folds = [0, 1]
    
    debug = False
    debug_count = 5

    if debug:
        test_soundscapes = '/kaggle/input/birdclef-2025/train_soundscapes'

class BirdCLEF2025Pipeline:
    class BirdCLEFModel(nn.Module):
        def __init__(self, cfg, num_classes, model_name):
            super().__init__()
            self.cfg = cfg
            self.model_name = model_name
            self.backbone = timm.create_model(
                model_name,
                pretrained=False,
                in_chans=cfg.in_channels,
                drop_rate=0.0,
                drop_path_rate=0.0
            )
            if 'efficientnet' in model_name:
                backbone_out = self.backbone.classifier.in_features
                self.backbone.classifier = nn.Identity()
            elif 'resnet' in model_name:
                backbone_out = self.backbone.fc.in_features
                self.backbone.fc = nn.Identity()
            else:
                backbone_out = self.backbone.get_classifier().in_features
                self.backbone.reset_classifier(0, '')
            
            self.pooling = nn.AdaptiveAvgPool2d(1)
            self.feat_dim = backbone_out
            self.classifier = nn.Linear(backbone_out, num_classes)
            
        def forward(self, x):
            features = self.backbone(x)
            if isinstance(features, dict):
                features = features['features']
            if len(features.shape) == 4:
                features = self.pooling(features)
                features = features.view(features.size(0), -1)
            logits = self.classifier(features)
            return logits

    def __init__(self, cfg):
        self.cfg = cfg
        self.taxonomy_df = None
        self.species_ids = []
        self.models = []
        self._load_taxonomy()

    def _load_taxonomy(self):
        print("Đang tải dữ liệu taxonomy...")
        self.taxonomy_df = pd.read_csv(self.cfg.taxonomy_csv)
        self.species_ids = self.taxonomy_df['primary_label'].tolist()
        print(f"Số lớp: {len(self.species_ids)}")

    def audio2melspec(self, audio_data):
        if np.isnan(audio_data).any():
            mean_signal = np.nanmean(audio_data)
            audio_data = np.nan_to_num(audio_data, nan=mean_signal)
        
        mel_spec = librosa.feature.melspectrogram(
            y=audio_data,
            sr=self.cfg.sample_rate,
            n_fft=self.cfg.n_fft,
            hop_length=self.cfg.hop_length,
            n_mels=self.cfg.n_mels,
            fmin=self.cfg.fmin,
            fmax=self.cfg.fmax,
            power=self.cfg.power,
            norm=self.cfg.norm,
            pad_mode=self.cfg.pad_mode
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max, top_db=self.cfg.top_db)
        mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        
        mel_spec_rgb = np.stack([mel_spec_norm] * 3, axis=-1)  # [H, W, 3]
        return mel_spec_rgb

    def process_audio_segment(self, audio_data):
        if len(audio_data) < self.cfg.sample_rate * self.cfg.WINDOW_SIZE:
            audio_data = np.pad(
                audio_data,
                (0, self.cfg.sample_rate * self.cfg.WINDOW_SIZE - len(audio_data)),
                mode='constant'
            )
        
        mel_spec_rgb = self.audio2melspec(audio_data)  # [H, W, 3]
        
        mel_spec_rgb = torch.from_numpy(mel_spec_rgb).float().permute(2, 0, 1)  # [3, H, W]
        mel_spec_rgb = transforms_v2.functional.resize(
            mel_spec_rgb,
            self.cfg.target_shape,
            interpolation=transforms_v2.InterpolationMode.BILINEAR
        )  # [3, H, W]
        mel_spec_rgb = mel_spec_rgb.permute(1, 2, 0).numpy()  # [H, W, 3]
        
        if mel_spec_rgb.shape[:2] != self.cfg.target_shape:
            logging.warning(f"Mel-spectrogram shape {mel_spec_rgb.shape[:2]} does not match target {self.cfg.target_shape}")
            
        return mel_spec_rgb.astype(np.float32)

    def find_model_files(self):
        model_files = [
            self.cfg.model_path1,  # /kaggle/input/modelwithunlabeldata/efficientnet_b0_best (2).pth
            self.cfg.model_path2   # /kaggle/input/datacuatuantran/efficientnet_b0.pth
        ]
        return [f for f in model_files if os.path.exists(f)]

    def load_models(self):
        self.models = []
        model_files = self.find_model_files()
        if not model_files:
            print(f"Cảnh báo: Không tìm thấy file mô hình trong {self.cfg.model_path1} hoặc {self.cfg.model_path2}!")
            return self.models

        print(f"Tìm thấy {len(model_files)} file mô hình.")
        
        for model_path in model_files:
            try:
                model_name = 'efficientnet_b0'  # Cả hai mô hình đều là efficientnet_b0
                print(f"Đang tải mô hình: {model_path} ({model_name})")
                checkpoint = torch.load(model_path, map_location=self.cfg.device)
                
                model = self.BirdCLEFModel(self.cfg, len(self.species_ids), model_name)
                
                if 'model_state_dict' in checkpoint:
                    model.load_state_dict(checkpoint['model_state_dict'])
                else:
                    model.load_state_dict(checkpoint)
                
                model = model.to(self.cfg.device)
                model.eval()
                self.models.append(model)
            except Exception as e:
                print(f"Lỗi khi tải mô hình {model_path}: {e}")
        
        return self.models

    def apply_tta(self, spec, tta_idx):
        if tta_idx == 0:
            return spec
        elif tta_idx == 1:
            return np.flip(spec, axis=1)
        elif tta_idx == 2:
            return np.flip(spec, axis=0)
        else:
            return spec

    def predict_on_spectrogram(self, audio_path):
        predictions = []
        row_ids = []
        soundscape_id = Path(audio_path).stem
        
        try:
            print(f"Đang xử lý {soundscape_id}...")
            audio_data, _ = librosa.load(audio_path, sr=self.cfg.sample_rate)
            total_segments = int(len(audio_data) / (self.cfg.sample_rate * self.cfg.WINDOW_SIZE))
            
            for segment_idx in range(total_segments):
                start_sample = segment_idx * self.cfg.sample_rate * self.cfg.WINDOW_SIZE
                end_sample = start_sample + self.cfg.sample_rate * self.cfg.WINDOW_SIZE
                segment_audio = audio_data[start_sample:end_sample]
                
                end_time_sec = (segment_idx + 1) * self.cfg.WINDOW_SIZE
                row_id = f"{soundscape_id}_{end_time_sec}"
                row_ids.append(row_id)

                if self.cfg.use_tta:
                    all_preds = []
                    for tta_idx in range(self.cfg.tta_count):
                        mel_spec_rgb = self.process_audio_segment(segment_audio)
                        mel_spec_rgb = self.apply_tta(mel_spec_rgb, tta_idx)
                        mel_spec_tensor = torch.tensor(mel_spec_rgb, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
                        mel_spec_tensor = mel_spec_tensor.to(self.cfg.device)

                        segment_preds = []
                        for model in self.models:
                            with torch.no_grad():
                                outputs = model(mel_spec_tensor)
                                probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                                segment_preds.append(probs)
                        avg_preds = np.mean(segment_preds, axis=0)  # Trung bình hai mô hình
                        all_preds.append(avg_preds)
                    final_preds = np.mean(all_preds, axis=0)  # Trung bình qua TTA
                else:
                    mel_spec_rgb = self.process_audio_segment(segment_audio)
                    mel_spec_tensor = torch.tensor(mel_spec_rgb, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
                    mel_spec_tensor = mel_spec_tensor.to(self.cfg.device)
                    
                    segment_preds = []
                    for model in self.models:
                        with torch.no_grad():
                            outputs = model(mel_spec_tensor)
                            probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                            segment_preds.append(probs)
                    final_preds = np.mean(segment_preds, axis=0)  # Trung bình hai mô hình
                
                predictions.append(final_preds)
        except Exception as e:
            print(f"Lỗi khi xử lý {audio_path}: {e}")
        
        return row_ids, predictions

    def run_inference(self):
        test_files = list(Path(self.cfg.test_soundscapes).glob('*.ogg'))
        if self.cfg.debug:
            print(f"Chế độ debug: Chỉ dùng {self.cfg.debug_count} file")
            test_files = test_files[:self.cfg.debug_count]
        print(f"Tìm thấy {len(test_files)} file test soundscapes")

        all_row_ids = []
        all_predictions = []

        for audio_path in tqdm(test_files):
            row_ids, predictions = self.predict_on_spectrogram(str(audio_path))
            all_row_ids.extend(row_ids)
            all_predictions.extend(predictions)
        
        return all_row_ids, all_predictions

    def create_submission(self, row_ids, predictions):
        print("Đang tạo dataframe submission...")
        submission_dict = {'row_id': row_ids}
        for i, species in enumerate(self.species_ids):
            submission_dict[species] = [pred[i] for pred in predictions]

        submission_df = pd.DataFrame(submission_dict)
        submission_df.set_index('row_id', inplace=True)

        sample_sub = pd.read_csv(self.cfg.submission_csv, index_col='row_id')
        missing_cols = set(sample_sub.columns) - set(submission_df.columns)
        if missing_cols:
            print(f"Cảnh báo: Thiếu {len(missing_cols)} loài trong submission")
            for col in missing_cols:
                submission_df[col] = 0.0

        submission_df = submission_df[sample_sub.columns]
        submission_df = submission_df.reset_index()
        
        return submission_df

    def smooth_submission(self, submission_path):
        print("Đang làm mượt kết quả submission...")
        sub = pd.read_csv(submission_path)
        cols = sub.columns[1:]
        groups = sub['row_id'].str.rsplit('_', n=1).str[0].values
        unique_groups = np.unique(groups)
        
        for group in unique_groups:
            idx = np.where(groups == group)[0]
            sub_group = sub.iloc[idx].copy()
            predictions = sub_group[cols].values
            new_predictions = predictions.copy()
            
            if predictions.shape[0] > 1:
                new_predictions[0] = (predictions[0] * 0.8) + (predictions[1] * 0.2)
                new_predictions[-1] = (predictions[-1] * 0.8) + (predictions[-2] * 0.2)
                for i in range(1, predictions.shape[0]-1):
                    new_predictions[i] = (predictions[i-1] * 0.2) + (predictions[i] * 0.6) + (predictions[i+1] * 0.2)
            sub.iloc[idx, 1:] = new_predictions
        
        sub.to_csv(submission_path, index=False)
        print(f"Đã lưu submission mượt tại {submission_path}")

    def run(self):
        start_time = time.time()
        print("Bắt đầu suy luận BirdCLEF-2025...")
        print(f"TTA: {self.cfg.use_tta} (số lần: {self.cfg.tta_count if self.cfg.use_tta else 0})")
    
        self.load_models()
        if not self.models:
            print("Không tìm thấy mô hình! Vui lòng kiểm tra đường dẫn mô hình.")
            return
    
        print(f"Sử dụng: {'một mô hình' if len(self.models) == 1 else f'ensemble {len(self.models)} mô hình'}")
        row_ids, predictions = self.run_inference()
        submission_df = self.create_submission(row_ids, predictions)
    
        submission_path = 'submission.csv'
        submission_df.to_csv(submission_path, index=False)
        print(f"Đã lưu file submission ban đầu tại {submission_path}")
    
        self.smooth_submission(submission_path)
    
        end_time = time.time()
        print(f"Hoàn thành suy luận (thời gian: {(end_time - start_time) / 60:.2f} phút)")

if __name__ == "__main__":
    cfg = CFG()
    print(f"Sử dụng thiết bị: {cfg.device}")
    pipeline = BirdCLEF2025Pipeline(cfg)
    pipeline.run()

In [ ]:
df = pd.read_csv("/kaggle/working/submission.csv")

In [ ]:
df.head()